# P16 mechanism controls
One change each from G3 (P13 prompt, retry budget 3). G3t: the guide sees a goal-only task (no feature or preprocessing instructions). G3d: the guide also gets a fixed data profile. '-p': P13 wrong lessons (drop a useful column, against the task). '-pc': wrong lessons that agree with the task's wording. E-pc: embedding top-3 on the 'pc' store. Predictions: `quick_runs/predictions/P16.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
T16 = [json.loads(l) for l in open(REPO / "quick_runs/results/P16/guide_trace.jsonl")]
for t in T16:  # embedding-only arms have no guide: seen = accepted = shown
    t.setdefault("seen", t["shown"]); t.setdefault("accepted", t["shown"])
print(Counter(t["arm"] for t in T16))
# Fix lessons and the contradicting TF03 lesson (runs out of 5); P13 G3 and P14 G3o (same prompt) for reference
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T16, "G3t"), stage1(T16, "G3d")]).set_index("arm")
# Wrong lessons shown (runs out of 20). -p: P13 wrong lessons; -pc: instruction-agreeing wrong lessons
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T16, "G3t-p"), wrong(T16, "G3d-p"),
                   wrong(T16, "E-pc"), wrong(T16, "G3-pc"), wrong(T16, "G3d-pc")]).set_index("arm")
print(s1.to_string()); wr

Counter({'G3t': 30, 'G3d': 30, 'G3t-p': 20, 'G3-pc': 20, 'E-pc': 20, 'G3d-p': 20, 'G3d-pc': 20})
     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3t                 5                  5          5                 5                  5          5                  5
G3d                 5                  0          5                 5                  5          5                  0


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3t-p,20,19,19,17,5,5,2,5
G3d-p,20,20,1,1,1,0,0,0
E-pc,20,20,20,20,5,5,5,5
G3-pc,20,20,20,20,5,5,5,5
G3d-pc,20,20,8,8,0,0,3,5


In [6]:
# Reasons for rejecting fix lessons: share citing the task's instructions (pre-set regex + contradict-only rule from REVIEW_CHECKS)
NOC = r"task (says|asks|states|requires|instructs)|instruction|use all|all available"
TASKWORD = r"prompt|task|requirement|constraint"
def cites_instr(reason):
    if not re.search(INSTR, reason, re.I): return False
    return bool(re.search(NOC, reason, re.I) or re.search(TASKWORD, reason, re.I))
rows = []
for t in T14 + T16:
    if t["arm"] not in ("G3o", "G3o-p", "G3t", "G3t-p", "G3d", "G3d-p", "G3-pc", "G3d-pc"): continue
    for r in t["rounds"]:
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            kind = "fix" if i in FIX.get(t["task"], set()) else "wrong" if i in WRONG_IDS else None
            if kind is None or i not in r["candidates"]: continue
            reason = str(j.get("reason", ""))
            rows.append({"arm": t["arm"], "task": t["task"][:7], "run": t["run"], "id": i, "kind": kind,
                         "useful": j.get("useful") is True, "instr": cites_instr(reason), "reason": reason})
jr = pd.DataFrame(rows)
rej = jr[~jr.useful]
print(rej.groupby(["arm", "kind"]).instr.agg(["sum", "count", "mean"]).round(2).to_string())
ex = rej.groupby(["arm", "kind", "task"]).head(1)
for _, r in ex.iterrows(): print(f"{r.arm} {r.kind} {r.task} [{r.id}]: {r.reason[:250]}")
print("accepted wrong lessons, examples:")
for _, r in jr[jr.useful & (jr.kind == "wrong")].groupby(["arm", "task"]).head(1).iterrows(): print(f"{r.arm} {r.task} [{r.id}]: {r.reason[:250]}")

              sum  count  mean
arm    kind                   
G3-pc  fix     23     32  0.72
       wrong    2     21  0.10
G3d    fix     18     36  0.50
G3d-p  fix      8     28  0.29
       wrong   22     38  0.58
G3d-pc fix      8     20  0.40
       wrong    5     53  0.09
G3o    fix     39     47  0.83
G3o-p  fix     35     39  0.90
       wrong   27     30  0.90
G3t    fix      6     38  0.16
G3t-p  fix      2     21  0.10
       wrong    6     18  0.33
G3o fix sf_tf03 [135]: This note suggests using domain reasoning to exclude some numerical features. This contradicts the explicit instruction to 'Keep only numerical feature columns'.
G3o fix sf_tf05 [164]: This note advises removing 'leakage' features (like expenditure or majorcards), which contradicts the explicit instruction to 'Use all available feature columns'.
G3o-p wrong sf_tf02 [711]: This note explicitly conflicts with the task instructions, which state to 'use all columns except player_id as features'.
G3o-p fix sf_tf

In [7]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P13_E", "P13_G3", "P16_G3t", "P16_G3d", "P13_Ep", "P13_G3p", "P16_G3tp", "P16_G3dp", "P16_Epc", "P16_G3pc", "P16_G3dpc"]
NAME = dict(zip(ARMS, ["A0 none", "C1 own", "E emb", "G3", "G3t blind", "G3d profile", "E-p", "G3-p", "G3t-p", "G3d-p", "E-pc", "G3-pc", "G3d-pc"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P16_G3t': (30, {'final_answer': 28, 'max_steps': 2}), 'P16_G3d': (30, {'final_answer': 28, 'max_steps': 2}), 'P13_Ep': (20, {'final_answer': 19, 'max_steps': 1}), 'P13_G3p': (20, {'final_answer': 20}), 'P16_G3tp': (20, {'final_answer': 17, 'max_steps': 3}), 'P16_G3dp': (20, {'final_answer': 20}), 'P16_Epc': (20, {'final_answer': 20}), 'P16_G3pc': (20, {'final_answer': 20}), 'P16_G3dpc': (20, {'final_answer': 20})}
pooled: {'A0 none': 0.075, 'C1 own': 0.75, 'E emb': 0.475, 'G3': 0.5, 'G3t blind': 0.425, 'G3d profile': 0.6, 'E-p': 0.425, 'G3-p': 0.45, 'G3t-p': 0.425, 'G3d-p': 0.6, 'E-pc': 0.025, 'G3-pc': 0.0, 'G3d-pc': 0.475}


,task,check,A0 none,C1 own,E emb,G3,G3t blind,G3d profile,E-p,G3-p,G3t-p,G3d-p,E-pc,G3-pc,G3d-pc
0,sf_tf01,checks_class_distribution,0,5,5,4,4,2,3,3,2,3,1,0,2
1,sf_tf01,uses_class_weight,0,5,5,5,5,5,5,5,5,5,0,0,5
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,5,5,5,5,5,4,5,0,0,5
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,4,3,3,4,5,2,5,0,0,5
5,sf_tf03,column_x_included,0,5,0,1,0,5,0,0,1,4,0,0,1
6,sf_tf03,applies_causal_reasoning,0,5,0,1,0,4,0,0,3,2,0,0,1
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,0,0,0,0,0,0


In [8]:
# TF05 leaking columns kept (of 4) per run
def leak_kept(a, r):
    j = json.load(open(RUNS / a / r["task_id"] / r["run"] / "judge.json"))
    return len(Counter(tuple(sorted(d.get("leakage_features_used") or [])) for d in j["draws"]).most_common(1)[0][0])
tf05_kept = {NAME[a]: [leak_kept(a, r) for r in sorted(recs[a], key=lambda r: r["run"]) if r["task_id"] == "sf_tf05_credit_card_leakage"] for a in ARMS}
tf05_kept

{'A0 none': [4, 4, 4, 4, 4],
 'C1 own': [1, 0, 1, 1, 1],
 'E emb': [4, 2, 2, 4, 4],
 'G3': [4, 4, 4, 4, 4],
 'G3t blind': [4, 4, 4, 4, 4],
 'G3d profile': [4, 4, 4, 4, 4],
 'E-p': [4, 4, 4, 4, 4],
 'G3-p': [4, 4, 4, 4, 4],
 'G3t-p': [2, 4, 4, 4, 2],
 'G3d-p': [4, 4, 4, 4, 4],
 'E-pc': [4, 4, 4, 4, 4],
 'G3-pc': [4, 4, 4, 4, 4],
 'G3d-pc': [4, 4, 4, 4, 4]}

In [9]:
# Trust funnel. -p arms: named column left out of the final model (executed variables). -pc arms: the lesson's advice
# followed = the targeted check fails (checks are stored as pass/fail: TF01 no class weights; TF02 unit not stripped;
# TF03 column_x used; TF05 a leaking column used)
TR = {(t["arm"], t["task"], t["run"]): t for t in T13 + T16}
def uses(r, col):
    v = r["executed"].get("variables") or []
    return any(col == x or x.startswith(col + "_") for x in v)
FOLLOW_C = {"sf_tf01_goal_imbalance": ("uses_class_weight", False), "sf_tf02_player_rep_norm": ("handles_unit_suffix_in_height", False),
            "sf_tf03_correlation_causation": ("column_x_included", False), "sf_tf05_credit_card_leakage": ("used_leakage_features", False)}
GARM = {"P13_G3p": "G3-p", "P16_G3tp": "G3t-p", "P16_G3dp": "G3d-p", "P16_Epc": "E-pc", "P16_G3pc": "G3-pc", "P16_G3dpc": "G3d-pc", "P13_Ep": "E-p"}
rows = []
for a, g in GARM.items():
    for r in recs[a]:
        t = TR.get((g, r["task_id"], r["run"]))
        if t is None: continue
        shown = bool(set(t["shown"]) & WRONG_IDS)
        if g.endswith("-pc"):
            key, val = FOLLOW_C[r["task_id"]]; followed = r["checks"].get(key) is val
        else:
            followed = not uses(r, WRONG_COL[r["task_id"]])
        rows.append({"arm": g, "task": r["task_id"][:7], "run": r["run"], "shown": shown, "followed": followed})
fun = pd.DataFrame(rows)
print(fun.groupby(["arm", "shown"]).followed.agg(["sum", "count"]).to_string())
fun.groupby(["arm", "task"])[["shown", "followed"]].sum()

              sum  count
arm    shown            
E-p    False    0     10
       True    10     10
E-pc   True    20     20
G3-p   False    0     15
       True     5      5
G3-pc  True    20     20
G3d-p  False    0     19
       True     1      1
G3d-pc False    1     12
       True     8      8
G3t-p  False    0      3
       True    10     17


shown  followed
arm    task                    
E-p    sf_tf01      5         5
       sf_tf02      0         0
       sf_tf03      0         0
       sf_tf05      5         5
E-pc   sf_tf01      5         5
       sf_tf02      5         5
       sf_tf03      5         5
       sf_tf05      5         5
G3-p   sf_tf01      4         4
       sf_tf02      0         0
       sf_tf03      0         0
       sf_tf05      1         1
G3-pc  sf_tf01      5         5
       sf_tf02      5         5
       sf_tf03      5         5
       sf_tf05      5         5
G3d-p  sf_tf01      1         1
       sf_tf02      0         0
       sf_tf03      0         0
       sf_tf05      0         0
G3d-pc sf_tf01      0         0
       sf_tf02      0         0
       sf_tf03      3         4
       sf_tf05      5         5
G3t-p  sf_tf01      5         5
       sf_tf02      5         0
       sf_tf03      2         2
       sf_tf05      5         3

In [10]:
# Harm and cost
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
gcost = {}
for arm, trs in [("G3 (P13)", [t for t in T13 if t["arm"] == "G3"]), ("G3t", [t for t in T16 if t["arm"] == "G3t"]), ("G3d", [t for t in T16 if t["arm"] == "G3d"])]:
    gcost[arm] = statistics.mean(sum(x["usage"]["prompt_tokens"] + x["usage"]["completion_tokens"] for x in t["rounds"] if x.get("usage")) for t in trs)
print("guide tokens per run:", {k: round(v) for k, v in gcost.items()}); scores

guide tokens per run: {'G3 (P13)': 6933, 'G3t': 8154, 'G3d': 8710}


arm,A0 none,C1 own,E emb,E-p,E-pc,G3,G3-p,G3-pc,G3d profile,G3d-p,G3d-pc,G3t blind,G3t-p
task,,,,,,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.765000e-01,7.713000e-01,7.387000e-01,7.600000e-01,7.715000e-01,7.449000e-01,7.865000e-01,7.658000e-01,7.574000e-01,7.659000e-01,7.713000e-01,7.269000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.657555e+06,4.664201e+06,4.650909e+06,4.650909e+06,4.686842e+06,4.655340e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.650909e+06
sf_tf03_correlation_causation,2.769462e+07,4.755266e+06,2.769464e+07,2.769464e+07,2.769463e+07,2.310314e+07,2.769458e+07,2.769463e+07,4.759155e+06,9.363499e+06,2.311203e+07,2.769464e+07,2.310365e+07
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,NaN,NaN,1.000000e+00,NaN,NaN,1.000000e+00,NaN,NaN,1.000000e+00,NaN
sf_tf05_credit_card_leakage,9.659000e-01,8.220000e-01,9.144000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.568000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.477000e-01,8.833000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.806000e-01,9.796000e-01,NaN,NaN,9.803000e-01,NaN,NaN,9.777000e-01,NaN,NaN,9.810000e-01,NaN


In [11]:
# Exact tests (two-sided Fisher), runs or check-runs as the unit
from scipy.stats import fisher_exact
passed = (pooled * 40).round().astype(int)
def fp(a, b, n): return round(fisher_exact([[a, n - a], [b, n - b]])[1], 4)
fisher = {"passed G3 vs G3d profile": fp(passed["G3"], passed["G3d profile"], 40),
          "passed G3-p vs G3d-p": fp(passed["G3-p"], passed["G3d-p"], 40),
          "passed G3-pc vs G3d-pc": fp(passed["G3-pc"], passed["G3d-pc"], 40),
          "passed E emb vs E-pc": fp(passed["E emb"], passed["E-pc"], 40),
          "wrong shown G3-p vs G3t-p": fp(int(wr.loc["G3-p", "wrong shown"]), int(wr.loc["G3t-p", "wrong shown"]), 20),
          "wrong shown G3-p vs G3-pc": fp(int(wr.loc["G3-p", "wrong shown"]), int(wr.loc["G3-pc", "wrong shown"]), 20),
          "wrong shown G3-pc vs G3d-pc": fp(int(wr.loc["G3-pc", "wrong shown"]), int(wr.loc["G3d-pc", "wrong shown"]), 20),
          "fix shown TF03+TF05 G3 vs G3t (of 10)": fp(1, 10, 10)}
fisher

{'passed G3 vs G3d profile': np.float64(0.5005),
 'passed G3-p vs G3d-p': np.float64(0.2629),
 'passed G3-pc vs G3d-pc': np.float64(0.0),
 'passed E emb vs E-pc': np.float64(0.0),
 'wrong shown G3-p vs G3t-p': np.float64(0.0003),
 'wrong shown G3-p vs G3-pc': np.float64(0.0),
 'wrong shown G3-pc vs G3d-pc': np.float64(0.0),
 'fix shown TF03+TF05 G3 vs G3t (of 10)': np.float64(0.0001)}

In [12]:
summary = {"stage1_fix": s1.reset_index().to_dict("records"), "stage1_wrong": wr.reset_index().to_dict("records"),
  "reasons_rejected_instr": rej.groupby(["arm", "kind"]).instr.agg(["sum", "count"]).reset_index().to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "tf05_leak_cols_kept": tf05_kept,
  "fisher_p": fisher, "trust": fun.groupby(["arm", "shown"]).followed.agg(["sum", "count"]).reset_index().to_dict("records"),
  "trust_by_task": fun.groupby(["arm", "task"])[["shown", "followed"]].sum().reset_index().to_dict("records"),
  "score_mean": scores.to_dict("index"), "guide_tokens_per_run": gcost}
json.dump(summary, open(REPO / "quick_runs/results/P16/summary.json", "w"), indent=1, default=str)
jr.to_csv(REPO / "quick_runs/results/P16/judgements.csv", index=False); fun.to_csv(REPO / "quick_runs/results/P16/trust_funnel.csv", index=False)
"done" 

'done'